In [2]:
# ============================================================
# PHISHGUARD
# Phishing Email Detection & Awareness Dashboard
# ============================================================
# SINGLE-CELL GOOGLE COLAB VERSION
#
# Defensive / educational project
# Uses synthetic and fictional data only.
#
# Features:
#   - Synthetic dataset generation
#   - Rule-based phishing detection
#   - Sender analysis
#   - Content analysis
#   - Static URL analysis
#   - Attachment filename analysis
#   - Explainable risk scoring
#   - Optional ML model
#   - SQLite analysis history
#   - Dashboard analytics
#   - Phishing awareness module
#   - Automated tests
#   - Professional high-contrast SOC dashboard
#
# SAFETY:
#   - Never sends emails
#   - Never visits URLs
#   - Never executes attachments
#   - Never collects credentials
#   - Uses fictional/reserved examples only
# ============================================================


# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import subprocess
import sys
import os
import re
import json
import sqlite3
import random
import string
import math
import html
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter

def install_packages():
    packages = [
        "gradio",
        "pandas",
        "numpy",
        "scikit-learn",
        "plotly",
        "joblib"
    ]

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + packages
    )

install_packages()


# ============================================================
# 2. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import gradio as gr
import plotly.graph_objects as go
import plotly.express as px
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 3. PROJECT DIRECTORIES
# ============================================================

BASE_DIR = Path("/content/phishguard")

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"
SCREENSHOT_DIR = BASE_DIR / "screenshots"
DOCS_DIR = BASE_DIR / "docs"
TEST_DIR = BASE_DIR / "tests"

for directory in [
    BASE_DIR,
    DATA_DIR,
    MODEL_DIR,
    REPORT_DIR,
    SCREENSHOT_DIR,
    DOCS_DIR,
    TEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


DATASET_PATH = DATA_DIR / "phishing_email_dataset.csv"
MODEL_PATH = MODEL_DIR / "phishguard_model.joblib"
DB_PATH = BASE_DIR / "phishguard.db"


# ============================================================
# 4. DATABASE
# ============================================================

def init_database():

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS analyses (
            analysis_id INTEGER PRIMARY KEY AUTOINCREMENT,
            sender TEXT,
            sender_domain TEXT,
            subject TEXT,
            risk_score INTEGER,
            classification TEXT,
            created_at TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS indicators (
            indicator_id INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id INTEGER,
            indicator_type TEXT,
            description TEXT,
            severity TEXT,
            FOREIGN KEY (analysis_id)
                REFERENCES analyses(analysis_id)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS url_analyses (
            url_analysis_id INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id INTEGER,
            url_safe_representation TEXT,
            risk_score INTEGER,
            findings TEXT,
            FOREIGN KEY (analysis_id)
                REFERENCES analyses(analysis_id)
        )
    """)

    conn.commit()
    conn.close()


init_database()


# ============================================================
# 5. SYNTHETIC DATASET GENERATOR
# ============================================================

LEGITIMATE_SENDERS = [
    "training@example.org",
    "hr@example.com",
    "projects@example.net",
    "newsletter@example.org",
    "support@example.com",
    "events@example.org",
    "admin@example.net",
    "library@example.org",
]

LEGITIMATE_SUBJECTS = [
    "Cybersecurity Workshop Reminder",
    "Project Meeting Reminder",
    "University Event Update",
    "Monthly Newsletter",
    "HR Policy Update",
    "Project Submission Information",
    "Team Meeting Schedule",
    "Training Session Confirmation",
    "Library Service Update",
    "Account Password Change Confirmation",
]

LEGITIMATE_BODIES = [
    "Hello, this is a reminder about the cybersecurity workshop scheduled for tomorrow. Please review the agenda before attending.",
    "Hello team, the project meeting is scheduled for tomorrow afternoon. Please bring your current progress update.",
    "This message provides information about the upcoming university event. Please review the schedule at your convenience.",
    "Thank you for subscribing to our monthly newsletter. Here are the latest updates and announcements.",
    "The HR department has published an updated policy document. Please review it through the normal internal portal.",
    "Your project submission schedule has been updated. Please review the official course portal for the latest information.",
    "This is a reminder about the upcoming team meeting. No action is required before the meeting.",
    "Your cybersecurity training session has been scheduled. Please attend at the assigned time.",
    "The library has updated its service schedule. Please check the official university information page.",
    "Your password was changed successfully. If you made this change, no further action is required.",
]

PHISHING_SENDERS = [
    "security-alert@example.net",
    "account-check.invalid.test",
    "verification-team.invalid.test",
    "billing-alert.example.net",
    "support-verification.invalid.test",
]

PHISHING_SUBJECTS = [
    "URGENT: Verify Your Account Immediately",
    "Your Account Will Be Suspended",
    "Immediate Payment Required",
    "You Have Won A Reward",
    "Password Expiration Warning",
    "Delivery Verification Required",
    "Urgent HR Document Request",
    "Invoice Payment Needed",
]

PHISHING_BODIES = [
    "URGENT. Your account will be suspended unless you verify your password immediately. Act now.",
    "Your account requires immediate verification. Confirm your password and personal information to avoid suspension.",
    "An outstanding payment requires immediate attention. Verify your billing information today.",
    "Congratulations. You have won a reward. Confirm your personal information to receive your prize.",
    "Your password will expire today. Verify your credentials immediately to prevent account suspension.",
    "Your delivery could not be completed. Confirm your information immediately to reschedule the delivery.",
    "This request is urgent. Please provide the requested employee information immediately.",
    "Your invoice is overdue. Confirm your payment details immediately to avoid further action.",
]

SAFE_URLS = [
    "https://example.org/workshop",
    "https://example.com/news",
    "https://example.net/project",
]

PHISHING_URLS = [
    "http://198.51.100.10/verify-account",
    "http://example.net/account/verify-password",
    "http://example.org/urgent-payment",
    "http://example.net/login/confirm",
]

ATTACHMENTS_SAFE = [
    "",
    "meeting_agenda.pdf",
    "project_notes.docx",
    "workshop_schedule.pdf",
]

ATTACHMENTS_RISKY = [
    "invoice.pdf.exe",
    "document.scr",
    "payment_details.js",
    "update.cmd",
]


def generate_synthetic_dataset(n=600):

    records = []

    for i in range(n):

        is_phishing = i % 2 == 0

        if not is_phishing:

            sender = random.choice(LEGITIMATE_SENDERS)
            subject = random.choice(LEGITIMATE_SUBJECTS)
            body = random.choice(LEGITIMATE_BODIES)

            url = random.choice(SAFE_URLS + [""])

            attachment = random.choice(ATTACHMENTS_SAFE)

            label = "LEGITIMATE"

        else:

            sender = random.choice(PHISHING_SENDERS)
            subject = random.choice(PHISHING_SUBJECTS)
            body = random.choice(PHISHING_BODIES)

            url = random.choice(PHISHING_URLS)

            attachment = random.choice(
                ATTACHMENTS_RISKY + ["", "invoice.pdf"]
            )

            label = "PHISHING"

        records.append({
            "email_id": f"EMAIL-{i+1:04d}",
            "sender": sender,
            "sender_domain": sender.split("@")[-1] if "@" in sender else "",
            "subject": subject,
            "body": body,
            "urls": url,
            "attachment_name": attachment,
            "label": label
        })

    df = pd.DataFrame(records)

    df.to_csv(DATASET_PATH, index=False)

    return df


if DATASET_PATH.exists():

    dataset = pd.read_csv(DATASET_PATH)

    if len(dataset) < 500:
        dataset = generate_synthetic_dataset()

else:
    dataset = generate_synthetic_dataset()


# ============================================================
# 6. DETECTION KEYWORDS
# ============================================================

URGENCY_KEYWORDS = [
    "urgent",
    "immediately",
    "immediate",
    "act now",
    "right now",
    "as soon as possible",
    "today",
    "within 24 hours",
]

CREDENTIAL_KEYWORDS = [
    "password",
    "verify your password",
    "login",
    "credentials",
    "otp",
    "one time password",
    "authentication",
    "sign in",
]

FINANCIAL_KEYWORDS = [
    "payment",
    "invoice",
    "bank",
    "billing",
    "refund",
    "money",
    "transfer",
    "payment details",
]

THREAT_KEYWORDS = [
    "suspended",
    "suspension",
    "blocked",
    "terminate",
    "penalty",
    "legal action",
    "locked",
    "disable",
]

REWARD_KEYWORDS = [
    "winner",
    "won",
    "prize",
    "reward",
    "gift",
    "bonus",
]

PERSONAL_INFO_KEYWORDS = [
    "personal information",
    "date of birth",
    "address",
    "phone number",
    "identity",
    "personal details",
]

SUSPICIOUS_URL_KEYWORDS = [
    "verify",
    "verification",
    "login",
    "password",
    "account",
    "secure",
    "payment",
    "confirm",
    "credential",
]

SHORTENER_PATTERNS = [
    "bit.ly",
    "tinyurl.com",
    "t.co",
    "is.gd",
    "ow.ly",
    "buff.ly",
]


# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

def safe_text(value):

    if value is None:
        return ""

    return str(value).strip()


def extract_domain(sender):

    sender = safe_text(sender).lower()

    if "@" not in sender:
        return ""

    return sender.split("@")[-1].strip()


def extract_urls(text):

    text = safe_text(text)

    pattern = r"https?://[^\s<>\"]+"

    return re.findall(pattern, text)


def keyword_count(text, keywords):

    text = safe_text(text).lower()

    return sum(text.count(keyword.lower()) for keyword in keywords)


def calculate_uppercase_ratio(text):

    letters = [c for c in text if c.isalpha()]

    if not letters:
        return 0

    uppercase = sum(c.isupper() for c in letters)

    return uppercase / len(letters)


# ============================================================
# 8. SENDER ANALYSIS
# ============================================================

def analyze_sender(sender, display_name=""):

    sender = safe_text(sender)
    display_name = safe_text(display_name)

    findings = []
    score = 0

    if not sender or "@" not in sender:

        findings.append({
            "type": "Sender",
            "description": "Sender address appears invalid or incomplete.",
            "severity": "medium"
        })

        score += 15

        return {
            "score": min(score, 100),
            "findings": findings
        }

    local, domain = sender.lower().split("@", 1)

    if len(domain) > 45:

        findings.append({
            "type": "Sender",
            "description": "Sender domain is unusually long.",
            "severity": "medium"
        })

        score += 8

    if domain.count(".") >= 3:

        findings.append({
            "type": "Sender",
            "description": "Sender domain contains multiple subdomain levels.",
            "severity": "medium"
        })

        score += 8

    if re.search(r"[0-9]{3,}", local):

        findings.append({
            "type": "Sender",
            "description": "Sender local-part contains an unusual numeric pattern.",
            "severity": "low"
        })

        score += 5

    if re.search(r"[^a-zA-Z0-9._%+-]", local):

        findings.append({
            "type": "Sender",
            "description": "Sender contains unusual characters.",
            "severity": "medium"
        })

        score += 7

    suspicious_terms = [
        "security",
        "verify",
        "verification",
        "alert",
        "billing",
        "support",
        "account"
    ]

    if any(term in local.lower() for term in suspicious_terms):

        findings.append({
            "type": "Sender",
            "description": "Sender name uses a security/account-related pattern.",
            "severity": "low"
        })

        score += 5

    if display_name:

        display_lower = display_name.lower()

        organization_words = [
            "bank",
            "university",
            "microsoft",
            "amazon",
            "google",
            "security",
            "hr",
            "finance"
        ]

        if any(word in display_lower for word in organization_words):

            if not any(
                word in domain
                for word in organization_words
            ):

                findings.append({
                    "type": "Sender",
                    "description": "Display name may not match the sender domain.",
                    "severity": "high"
                })

                score += 15

    if not findings:

        findings.append({
            "type": "Sender",
            "description": "No strong sender anomaly was identified.",
            "severity": "info"
        })

    return {
        "score": min(score, 100),
        "findings": findings
    }


# ============================================================
# 9. CONTENT ANALYSIS
# ============================================================

def analyze_email_content(subject, body):

    subject = safe_text(subject)
    body = safe_text(body)

    combined = f"{subject} {body}"

    findings = []
    score = 0

    urgency = keyword_count(combined, URGENCY_KEYWORDS)

    if urgency:

        findings.append({
            "type": "Urgency",
            "description": f"Urgency/manipulation language detected ({urgency} signal(s)).",
            "severity": "medium"
        })

        score += min(10, urgency * 3)

    credential = keyword_count(combined, CREDENTIAL_KEYWORDS)

    if credential:

        findings.append({
            "type": "Credential Request",
            "description": "Credential or authentication-related language detected.",
            "severity": "high"
        })

        score += min(20, credential * 8)

    financial = keyword_count(combined, FINANCIAL_KEYWORDS)

    if financial:

        findings.append({
            "type": "Financial Pressure",
            "description": "Payment or financial-pressure language detected.",
            "severity": "medium"
        })

        score += min(15, financial * 5)

    threat = keyword_count(combined, THREAT_KEYWORDS)

    if threat:

        findings.append({
            "type": "Threat/Fear",
            "description": "Threatening or fear-based language detected.",
            "severity": "medium"
        })

        score += min(15, threat * 5)

    reward = keyword_count(combined, REWARD_KEYWORDS)

    if reward:

        findings.append({
            "type": "Reward",
            "description": "Prize/reward-related language detected.",
            "severity": "medium"
        })

        score += min(10, reward * 5)

    personal = keyword_count(combined, PERSONAL_INFO_KEYWORDS)

    if personal:

        findings.append({
            "type": "Personal Information",
            "description": "Request for personal information detected.",
            "severity": "high"
        })

        score += min(15, personal * 6)

    if not subject.strip():

        findings.append({
            "type": "Subject",
            "description": "Email has an empty subject.",
            "severity": "low"
        })

        score += 3

    if not body.strip():

        findings.append({
            "type": "Body",
            "description": "Email body is empty.",
            "severity": "low"
        })

        score += 2

    generic_greetings = [
        "dear customer",
        "dear user",
        "dear sir",
        "dear madam",
        "hello customer",
        "dear account holder"
    ]

    if any(greeting in combined.lower() for greeting in generic_greetings):

        findings.append({
            "type": "Generic Greeting",
            "description": "Generic greeting detected.",
            "severity": "low"
        })

        score += 5

    exclamation_count = combined.count("!")

    if exclamation_count >= 3:

        findings.append({
            "type": "Formatting",
            "description": "Multiple exclamation marks indicate heightened emotional language.",
            "severity": "low"
        })

        score += 3

    uppercase_ratio = calculate_uppercase_ratio(combined)

    if uppercase_ratio > 0.35:

        findings.append({
            "type": "Formatting",
            "description": "High uppercase ratio detected.",
            "severity": "low"
        })

        score += 4

    if not findings:

        findings.append({
            "type": "Content",
            "description": "No strong social-engineering language detected.",
            "severity": "info"
        })

    return {
        "score": min(score, 100),
        "findings": findings
    }


# ============================================================
# 10. URL ANALYSIS
# ============================================================

def analyze_url(url):

    url = safe_text(url)

    findings = []
    score = 0

    if not url:

        return {
            "url": "",
            "score": 0,
            "findings": []
        }

    try:

        parsed = urlparse(url)

        scheme = parsed.scheme.lower()
        hostname = parsed.hostname or ""
        path = parsed.path or ""

        if scheme != "https":

            findings.append(
                "URL does not use HTTPS."
            )

            score += 15

        if re.fullmatch(
            r"\d{1,3}(\.\d{1,3}){3}",
            hostname
        ):

            findings.append(
                "URL uses a raw IP address."
            )

            score += 25

        if hostname.count(".") >= 3:

            findings.append(
                "URL contains multiple subdomains."
            )

            score += 10

        if len(url) > 120:

            findings.append(
                "URL is unusually long."
            )

            score += 8

        if any(
            keyword in url.lower()
            for keyword in SUSPICIOUS_URL_KEYWORDS
        ):

            findings.append(
                "URL contains account/security-related keywords."
            )

            score += 10

        if any(
            shortener in hostname.lower()
            for shortener in SHORTENER_PATTERNS
        ):

            findings.append(
                "URL resembles a shortening service."
            )

            score += 15

        if re.search(
            r"[@%]|//.*//",
            url
        ):

            findings.append(
                "URL contains an unusual structural pattern."
            )

            score += 10

        if not findings:

            findings.append(
                "No strong static URL anomaly detected."
            )

    except Exception:

        findings.append(
            "URL could not be parsed safely."
        )

        score += 15

    return {
        "url": url,
        "score": min(score, 100),
        "findings": findings
    }


def analyze_urls(text):

    urls = extract_urls(text)

    results = []

    for url in urls:

        results.append(
            analyze_url(url)
        )

    if not results:

        return {
            "score": 0,
            "results": []
        }

    maximum = max(
        result["score"]
        for result in results
    )

    return {
        "score": min(maximum, 100),
        "results": results
    }


# ============================================================
# 11. ATTACHMENT ANALYSIS
# ============================================================

DANGEROUS_EXTENSIONS = [
    ".exe",
    ".scr",
    ".bat",
    ".cmd",
    ".js",
    ".vbs",
    ".ps1",
    ".msi"
]


def analyze_attachment(filename):

    filename = safe_text(filename)

    findings = []
    score = 0

    if not filename:

        return {
            "score": 0,
            "findings": [
                {
                    "type": "Attachment",
                    "description": "No attachment filename supplied.",
                    "severity": "info"
                }
            ]
        }

    lower = filename.lower()

    if any(
        lower.endswith(ext)
        for ext in DANGEROUS_EXTENSIONS
    ):

        findings.append({
            "type": "Attachment",
            "description": "Potentially executable/script attachment extension detected.",
            "severity": "high"
        })

        score += 25

    double_extensions = [
        ".pdf.exe",
        ".doc.exe",
        ".docx.exe",
        ".xls.exe",
        ".jpg.exe",
        ".png.exe"
    ]

    if any(
        pattern in lower
        for pattern in double_extensions
    ):

        findings.append({
            "type": "Attachment",
            "description": "Double-extension attachment pattern detected.",
            "severity": "high"
        })

        score += 25

    if not findings:

        findings.append({
            "type": "Attachment",
            "description": "No obvious risky executable extension detected.",
            "severity": "info"
        })

    return {
        "score": min(score, 100),
        "findings": findings
    }


# ============================================================
# 12. STRUCTURED FEATURE EXTRACTION
# ============================================================

def extract_email_features(
    sender,
    subject,
    body,
    attachment=""
):

    content = analyze_email_content(
        subject,
        body
    )

    url_analysis = analyze_urls(
        f"{body} {subject}"
    )

    attachment_analysis = analyze_attachment(
        attachment
    )

    sender_analysis = analyze_sender(
        sender
    )

    urls = extract_urls(
        f"{body} {subject}"
    )

    combined = f"{subject} {body}"

    features = {

        "urgent_keyword_count":
            keyword_count(combined, URGENCY_KEYWORDS),

        "credential_keyword_count":
            keyword_count(combined, CREDENTIAL_KEYWORDS),

        "financial_keyword_count":
            keyword_count(combined, FINANCIAL_KEYWORDS),

        "threat_keyword_count":
            keyword_count(combined, THREAT_KEYWORDS),

        "url_count":
            len(urls),

        "suspicious_url_count":
            sum(
                1
                for result in url_analysis["results"]
                if result["score"] >= 20
            ),

        "has_ip_url":
            int(
                any(
                    re.fullmatch(
                        r"\d{1,3}(\.\d{1,3}){3}",
                        urlparse(url).hostname or ""
                    )
                    for url in urls
                )
            ),

        "has_shortened_url_pattern":
            int(
                any(
                    any(
                        shortener in url.lower()
                        for shortener in SHORTENER_PATTERNS
                    )
                    for url in urls
                )
            ),

        "sender_domain_length":
            len(extract_domain(sender)),

        "subdomain_count":
            extract_domain(sender).count("."),

        "suspicious_attachment":
            int(
                attachment_analysis["score"] >= 20
            ),

        "generic_greeting":
            int(
                any(
                    phrase in combined.lower()
                    for phrase in [
                        "dear customer",
                        "dear user",
                        "dear sir",
                        "dear madam"
                    ]
                )
            ),

        "contains_password_request":
            int(
                keyword_count(
                    combined,
                    CREDENTIAL_KEYWORDS
                ) > 0
            ),

        "contains_personal_info_request":
            int(
                keyword_count(
                    combined,
                    PERSONAL_INFO_KEYWORDS
                ) > 0
            ),

        "exclamation_count":
            combined.count("!"),

        "uppercase_ratio":
            calculate_uppercase_ratio(
                combined
            ),

        "body_length":
            len(body),

        "subject_length":
            len(subject)
    }

    return features


# ============================================================
# 13. RULE-BASED RISK ENGINE
# ============================================================

def calculate_phishing_score(
    sender_result,
    content_result,
    url_result,
    attachment_result
):

    score = 0
    reasons = []

    # Sender
    if sender_result["score"] > 0:

        contribution = min(
            15,
            sender_result["score"]
        )

        score += contribution

        reasons.append(
            f"Sender analysis contributed +{contribution}"
        )

    # Content
    content_score = content_result["score"]

    if content_score > 0:

        contribution = min(
            35,
            content_score
        )

        score += contribution

        reasons.append(
            f"Content analysis contributed +{contribution}"
        )

    # URL
    url_score = url_result["score"]

    if url_score > 0:

        contribution = min(
            25,
            url_score
        )

        score += contribution

        reasons.append(
            f"URL analysis contributed +{contribution}"
        )

    # Attachment
    attachment_score = attachment_result["score"]

    if attachment_score > 0:

        contribution = min(
            25,
            attachment_score
        )

        score += contribution

        reasons.append(
            f"Attachment analysis contributed +{contribution}"
        )

    score = min(
        100,
        int(score)
    )

    if score <= 20:

        classification = "LOW RISK"

    elif score <= 40:

        classification = "MODERATE RISK"

    elif score <= 70:

        classification = "SUSPICIOUS"

    else:

        classification = "HIGH RISK / LIKELY PHISHING"

    return {
        "score": score,
        "classification": classification,
        "reasons": reasons
    }


# ============================================================
# 14. ML MODEL
# ============================================================

ML_AVAILABLE = False
ML_MODEL = None
ML_VECTORIZER = None
ML_METRICS = {}


def train_ml_model():

    global ML_AVAILABLE
    global ML_MODEL
    global ML_VECTORIZER
    global ML_METRICS

    df = dataset.copy()

    df["text"] = (
        df["subject"].fillna("") +
        " " +
        df["body"].fillna("")
    )

    X = df["text"]
    y = (
        df["label"]
        .map({
            "LEGITIMATE": 0,
            "PHISHING": 1
        })
        .astype(int)
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=1
    )

    X_train_vec = vectorizer.fit_transform(
        X_train
    )

    X_test_vec = vectorizer.transform(
        X_test
    )

    model = LogisticRegression(
        max_iter=500,
        class_weight="balanced"
    )

    model.fit(
        X_train_vec,
        y_train
    )

    predictions = model.predict(
        X_test_vec
    )

    metrics = {
        "accuracy":
            accuracy_score(
                y_test,
                predictions
            ),

        "precision":
            precision_score(
                y_test,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_test,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_test,
                predictions,
                zero_division=0
            )
    }

    bundle = {
        "model": model,
        "vectorizer": vectorizer
    }

    joblib.dump(
        bundle,
        MODEL_PATH
    )

    ML_MODEL = model
    ML_VECTORIZER = vectorizer
    ML_METRICS = metrics
    ML_AVAILABLE = True

    return metrics


try:
    train_ml_model()
except Exception as e:
    print("ML training warning:", e)


# ============================================================
# 15. COMPLETE EMAIL ANALYSIS
# ============================================================

def analyze_email(
    sender,
    subject,
    body,
    attachment=""
):

    sender = safe_text(sender)
    subject = safe_text(subject)
    body = safe_text(body)
    attachment = safe_text(attachment)

    sender_result = analyze_sender(
        sender
    )

    content_result = analyze_email_content(
        subject,
        body
    )

    url_result = analyze_urls(
        f"{subject} {body}"
    )

    attachment_result = analyze_attachment(
        attachment
    )

    risk_result = calculate_phishing_score(
        sender_result,
        content_result,
        url_result,
        attachment_result
    )

    all_findings = (
        sender_result["findings"] +
        content_result["findings"] +
        attachment_result["findings"]
    )

    for url_data in url_result["results"]:

        for finding in url_data["findings"]:

            all_findings.append({
                "type": "URL",
                "description": finding,
                "severity": (
                    "high"
                    if url_data["score"] >= 40
                    else "medium"
                )
            })

    # --------------------------------------------------------
    # Save metadata only
    # --------------------------------------------------------

    conn = sqlite3.connect(
        DB_PATH
    )

    cursor = conn.cursor()

    sender_domain = extract_domain(
        sender
    )

    created_at = datetime.now().isoformat(
        timespec="seconds"
    )

    cursor.execute(
        """
        INSERT INTO analyses
        (
            sender,
            sender_domain,
            subject,
            risk_score,
            classification,
            created_at
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            sender,
            sender_domain,
            subject,
            risk_result["score"],
            risk_result["classification"],
            created_at
        )
    )

    analysis_id = cursor.lastrowid

    for finding in all_findings:

        cursor.execute(
            """
            INSERT INTO indicators
            (
                analysis_id,
                indicator_type,
                description,
                severity
            )
            VALUES (?, ?, ?, ?)
            """,
            (
                analysis_id,
                finding["type"],
                finding["description"],
                finding["severity"]
            )
        )

    for url_data in url_result["results"]:

        cursor.execute(
            """
            INSERT INTO url_analyses
            (
                analysis_id,
                url_safe_representation,
                risk_score,
                findings
            )
            VALUES (?, ?, ?, ?)
            """,
            (
                analysis_id,
                url_data["url"],
                url_data["score"],
                json.dumps(
                    url_data["findings"]
                )
            )
        )

    conn.commit()
    conn.close()

    return {
        "analysis_id": analysis_id,
        "sender": sender,
        "subject": subject,
        "sender_result": sender_result,
        "content_result": content_result,
        "url_result": url_result,
        "attachment_result": attachment_result,
        "risk_result": risk_result,
        "features": extract_email_features(
            sender,
            subject,
            body,
            attachment
        ),
        "findings": all_findings
    }


# ============================================================
# 16. DEMO EMAILS
# ============================================================

PHISHING_DEMO = {
    "sender":
        "security-alert@account-check.invalid.test",

    "subject":
        "URGENT: Verify Your Account Immediately",

    "body":
        """URGENT: Your account will be suspended today.

Please verify your password and personal information immediately
to prevent account suspension.

Confirm your account using:
http://198.51.100.10/verify-account

Failure to act immediately may result in your account being blocked.""",

    "attachment":
        "invoice.pdf.exe"
}


LEGITIMATE_DEMO = {
    "sender":
        "training@example.org",

    "subject":
        "Cybersecurity Workshop Reminder",

    "body":
        """Hello,

This is a reminder about the cybersecurity workshop scheduled
for tomorrow.

The workshop will cover phishing awareness, password hygiene,
social engineering, and safe browsing practices.

No account verification or payment is required.

Thank you.""",

    "attachment":
        "workshop_schedule.pdf"
}


# ============================================================
# 17. DASHBOARD DATA
# ============================================================

def get_dashboard_dataframe():

    conn = sqlite3.connect(
        DB_PATH
    )

    df = pd.read_sql_query(
        """
        SELECT
            analysis_id,
            sender_domain,
            subject,
            risk_score,
            classification,
            created_at
        FROM analyses
        ORDER BY analysis_id DESC
        """,
        conn
    )

    conn.close()

    return df


def dashboard_stats():

    df = get_dashboard_dataframe()

    if df.empty:

        return {
            "total": 0,
            "high": 0,
            "suspicious": 0,
            "low": 0,
            "average": 0
        }

    return {
        "total":
            len(df),

        "high":
            int(
                (
                    df["classification"]
                    == "HIGH RISK / LIKELY PHISHING"
                ).sum()
            ),

        "suspicious":
            int(
                (
                    df["classification"]
                    == "SUSPICIOUS"
                ).sum()
            ),

        "low":
            int(
                (
                    df["classification"]
                    == "LOW RISK"
                ).sum()
            ),

        "average":
            round(
                df["risk_score"].mean(),
                1
            )
    }


# ============================================================
# 18. PROFESSIONAL HTML HELPERS
# ============================================================

def escape(value):

    return html.escape(
        safe_text(value)
    )


def severity_badge(severity):

    severity = safe_text(
        severity
    ).lower()

    if severity == "high":

        return (
            '<span class="severity-badge high">'
            'HIGH'
            '</span>'
        )

    if severity == "medium":

        return (
            '<span class="severity-badge medium">'
            'MEDIUM'
            '</span>'
        )

    if severity == "low":

        return (
            '<span class="severity-badge low">'
            'LOW'
            '</span>'
        )

    return (
        '<span class="severity-badge info">'
        'INFO'
        '</span>'
    )


def risk_classification_html(
    classification
):

    if "HIGH RISK" in classification:

        return (
            '<div class="classification high">'
            f'{escape(classification)}'
            '</div>'
        )

    if classification == "SUSPICIOUS":

        return (
            '<div class="classification suspicious">'
            f'{escape(classification)}'
            '</div>'
        )

    if classification == "MODERATE RISK":

        return (
            '<div class="classification moderate">'
            f'{escape(classification)}'
            '</div>'
        )

    return (
        '<div class="classification low">'
        f'{escape(classification)}'
        '</div>'
    )


# ============================================================
# 19. ANALYSIS RESULT HTML
# ============================================================

def format_analysis_result(result):

    risk = result["risk_result"]

    score = risk["score"]
    classification = risk["classification"]

    sender_score = result["sender_result"]["score"]
    content_score = result["content_result"]["score"]
    url_score = result["url_result"]["score"]
    attachment_score = result["attachment_result"]["score"]

    finding_rows = ""

    for finding in result["findings"]:

        finding_rows += f"""
        <div class="finding-row">

            <div class="finding-left">

                <div class="finding-icon">
                    !
                </div>

                <div>

                    <div class="finding-type">
                        {escape(finding["type"])}
                    </div>

                    <div class="finding-description">
                        {escape(finding["description"])}
                    </div>

                </div>

            </div>

            {severity_badge(finding["severity"])}

        </div>
        """

    if not finding_rows:

        finding_rows = """
        <div class="empty-state">
            No significant indicators detected.
        </div>
        """

    url_html = ""

    for url in result["url_result"]["results"]:

        url_html += f"""
        <div class="url-card">

            <div class="url-address">
                {escape(url["url"])}
            </div>

            <div class="url-score">
                Static risk score:
                <strong>{url["score"]}/100</strong>
            </div>

            <ul>
        """

        for finding in url["findings"]:

            url_html += (
                f"<li>{escape(finding)}</li>"
            )

        url_html += """
            </ul>

        </div>
        """

    if not url_html:

        url_html = """
        <div class="empty-state">
            No URLs detected in the submitted email.
        </div>
        """

    recommendations = [
        "Do not click suspicious links.",
        "Do not open unexpected attachments.",
        "Verify the sender using a trusted channel.",
        "Use the organization's official website or application directly.",
        "Report suspicious messages to the appropriate security team."
    ]

    recommendation_html = "".join(
        f"""
        <div class="recommendation">
            <span class="check">✓</span>
            <span>{escape(item)}</span>
        </div>
        """
        for item in recommendations
    )

    return f"""

    <div class="result-container">

        <div class="risk-hero">

            <div>

                <div class="eyebrow">
                    PHISHGUARD ANALYSIS
                </div>

                <div class="risk-title">
                    Phishing Risk Assessment
                </div>

                <div class="risk-subtitle">
                    Explainable defensive analysis based on
                    sender, content, URL and attachment signals.
                </div>

            </div>

            <div class="risk-score-box">

                <div class="risk-score-label">
                    RISK SCORE
                </div>

                <div class="risk-score-number">
                    {score}
                    <span>/100</span>
                </div>

            </div>

        </div>


        <div class="classification-wrapper">

            {risk_classification_html(classification)}

        </div>


        <div class="mini-grid">

            <div class="mini-card">

                <div class="mini-label">
                    SENDER
                </div>

                <div class="mini-value">
                    {sender_score}/100
                </div>

            </div>

            <div class="mini-card">

                <div class="mini-label">
                    CONTENT
                </div>

                <div class="mini-value">
                    {content_score}/100
                </div>

            </div>

            <div class="mini-card">

                <div class="mini-label">
                    URL
                </div>

                <div class="mini-value">
                    {url_score}/100
                </div>

            </div>

            <div class="mini-card">

                <div class="mini-label">
                    ATTACHMENT
                </div>

                <div class="mini-value">
                    {attachment_score}/100
                </div>

            </div>

        </div>


        <div class="section-card">

            <div class="section-title">
                Why was this email flagged?
            </div>

            <div class="section-subtitle">
                These findings explain the signals contributing
                to the overall risk assessment.
            </div>

            {finding_rows}

        </div>


        <div class="section-card">

            <div class="section-title">
                URL Analysis
            </div>

            <div class="section-subtitle">
                URLs are analyzed statically. PHISHGUARD does not
                visit or execute suspicious destinations.
            </div>

            {url_html}

        </div>


        <div class="section-card recommendation-card">

            <div class="section-title">
                Recommended Actions
            </div>

            {recommendation_html}

        </div>


        <div class="analysis-disclaimer">

            <strong>Important:</strong>
            A risk score is not proof of malicious intent.
            Detection should support analyst judgment and
            contextual investigation.

        </div>

    </div>

    """


# ============================================================
# 20. ANALYZE BUTTON
# ============================================================

def analyze_for_dashboard(
    sender,
    subject,
    body,
    attachment
):

    if not sender and not subject and not body:

        return """
        <div class="empty-result">

            <div class="empty-icon">
                !
            </div>

            <div class="empty-title">
                Email content required
            </div>

            <div class="empty-text">
                Enter at least a sender, subject, or email body
                before starting an analysis.
            </div>

        </div>
        """

    result = analyze_email(
        sender,
        subject,
        body,
        attachment
    )

    return format_analysis_result(
        result
    )


# ============================================================
# 21. LOAD DEMO FUNCTIONS
# ============================================================

def load_phishing_demo():

    return (
        PHISHING_DEMO["sender"],
        PHISHING_DEMO["subject"],
        PHISHING_DEMO["body"],
        PHISHING_DEMO["attachment"]
    )


def load_legitimate_demo():

    return (
        LEGITIMATE_DEMO["sender"],
        LEGITIMATE_DEMO["subject"],
        LEGITIMATE_DEMO["body"],
        LEGITIMATE_DEMO["attachment"]
    )


# ============================================================
# 22. HISTORY TABLE
# ============================================================

def history_table():

    df = get_dashboard_dataframe()

    if df.empty:

        return pd.DataFrame(
            columns=[
                "ID",
                "Sender Domain",
                "Subject",
                "Risk",
                "Classification",
                "Analyzed At"
            ]
        )

    return df.rename(
        columns={
            "analysis_id": "ID",
            "sender_domain": "Sender Domain",
            "subject": "Subject",
            "risk_score": "Risk",
            "classification": "Classification",
            "created_at": "Analyzed At"
        }
    )


# ============================================================
# 23. ANALYTICS CHARTS
# ============================================================

def make_classification_chart():

    df = get_dashboard_dataframe()

    labels = [
        "LOW RISK",
        "MODERATE RISK",
        "SUSPICIOUS",
        "HIGH RISK / LIKELY PHISHING"
    ]

    values = []

    for label in labels:

        if df.empty:
            values.append(0)

        else:

            values.append(
                int(
                    (
                        df["classification"]
                        == label
                    ).sum()
                )
            )

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=values,
                text=values,
                textposition="outside"
            )
        ]
    )

    fig.update_layout(
        title="Classification Distribution",
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#F8FAFC",
            family="Inter, Arial, sans-serif"
        ),
        xaxis=dict(
            title="Classification",
            color="#CBD5E1",
            gridcolor="#26344A"
        ),
        yaxis=dict(
            title="Emails",
            color="#CBD5E1",
            gridcolor="#26344A"
        ),
        margin=dict(
            l=40,
            r=30,
            t=60,
            b=80
        )
    )

    return fig


def make_risk_chart():

    df = get_dashboard_dataframe()

    if df.empty:

        scores = [0]

    else:

        scores = df["risk_score"].tolist()

    fig = go.Figure(
        data=[
            go.Histogram(
                x=scores,
                nbinsx=10
            )
        ]
    )

    fig.update_layout(
        title="Risk Score Distribution",
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#F8FAFC",
            family="Inter, Arial, sans-serif"
        ),
        xaxis=dict(
            title="Risk Score",
            color="#CBD5E1",
            gridcolor="#26344A"
        ),
        yaxis=dict(
            title="Analyses",
            color="#CBD5E1",
            gridcolor="#26344A"
        )
    )

    return fig


def make_trend_chart():

    df = get_dashboard_dataframe()

    if df.empty:

        dates = []
        counts = []

    else:

        df["date"] = pd.to_datetime(
            df["created_at"]
        ).dt.date

        grouped = (
            df.groupby("date")
            .size()
            .reset_index(
                name="count"
            )
        )

        dates = grouped["date"]
        counts = grouped["count"]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=dates,
            y=counts,
            mode="lines+markers",
            name="Analyses"
        )
    )

    fig.update_layout(
        title="Analysis Trend",
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#F8FAFC",
            family="Inter, Arial, sans-serif"
        ),
        xaxis=dict(
            title="Date",
            color="#CBD5E1",
            gridcolor="#26344A"
        ),
        yaxis=dict(
            title="Analyses",
            color="#CBD5E1",
            gridcolor="#26344A"
        )
    )

    return fig


# ============================================================
# 24. DASHBOARD OVERVIEW
# ============================================================

def dashboard_overview():

    stats = dashboard_stats()

    return f"""

    <div class="overview-grid">

        <div class="kpi-card blue">

            <div class="kpi-label">
                TOTAL ANALYSES
            </div>

            <div class="kpi-number">
                {stats["total"]}
            </div>

            <div class="kpi-description">
                Emails analyzed
            </div>

        </div>


        <div class="kpi-card red">

            <div class="kpi-label">
                HIGH RISK
            </div>

            <div class="kpi-number">
                {stats["high"]}
            </div>

            <div class="kpi-description">
                Likely phishing
            </div>

        </div>


        <div class="kpi-card amber">

            <div class="kpi-label">
                SUSPICIOUS
            </div>

            <div class="kpi-number">
                {stats["suspicious"]}
            </div>

            <div class="kpi-description">
                Requires review
            </div>

        </div>


        <div class="kpi-card green">

            <div class="kpi-label">
                LOW RISK
            </div>

            <div class="kpi-number">
                {stats["low"]}
            </div>

            <div class="kpi-description">
                No strong indicators
            </div>

        </div>


        <div class="kpi-card purple">

            <div class="kpi-label">
                AVG RISK
            </div>

            <div class="kpi-number">
                {stats["average"]}
            </div>

            <div class="kpi-description">
                Average score / 100
            </div>

        </div>

    </div>

    """


# ============================================================
# 25. AWARENESS CONTENT
# ============================================================

AWARENESS_LESSONS = [

    (
        "01",
        "Check the Sender",
        "Do not rely only on a familiar display name. "
        "Inspect the actual sender address and domain."
    ),

    (
        "02",
        "Question Urgency",
        "Attackers often create pressure by claiming "
        "that an account will be suspended or a payment "
        "will fail unless you act immediately."
    ),

    (
        "03",
        "Inspect Links",
        "Before clicking, examine the destination domain. "
        "HTTPS provides encryption but does not guarantee "
        "that a website is trustworthy."
    ),

    (
        "04",
        "Protect Credentials",
        "Unexpected requests for passwords, OTPs or "
        "authentication information should receive "
        "extra scrutiny."
    ),

    (
        "05",
        "Treat Attachments Carefully",
        "Unexpected executable or script attachments can "
        "be dangerous. Never execute suspicious files."
    ),

    (
        "06",
        "Verify Through Another Channel",
        "If an email requests money, credentials or sensitive "
        "information, verify the request using a trusted "
        "communication channel."
    )

]


def awareness_html():

    cards = ""

    for number, title, description in AWARENESS_LESSONS:

        cards += f"""

        <div class="lesson-card">

            <div class="lesson-number">
                {number}
            </div>

            <div class="lesson-title">
                {escape(title)}
            </div>

            <div class="lesson-description">
                {escape(description)}
            </div>

        </div>

        """

    return f"""

    <div class="awareness-header">

        <div class="eyebrow">
            SECURITY AWARENESS
        </div>

        <div class="awareness-title">
            Before You Click
        </div>

        <div class="awareness-description">
            Small verification habits can prevent major
            security incidents.
        </div>

    </div>

    <div class="lesson-grid">

        {cards}

    </div>

    <div class="checklist-card">

        <div class="section-title">
            Quick Phishing Checklist
        </div>

        <div class="checklist-grid">

            <div>✓ Check the sender domain</div>
            <div>✓ Question unexpected urgency</div>
            <div>✓ Inspect links before clicking</div>
            <div>✓ Never share passwords by email</div>
            <div>✓ Be cautious with attachments</div>
            <div>✓ Verify unusual requests independently</div>
            <div>✓ Report suspicious emails</div>
            <div>✓ When uncertain, stop and verify</div>

        </div>

    </div>

    """


# ============================================================
# 26. PROJECT INFORMATION
# ============================================================

ABOUT_HTML = """

<div class="about-container">

    <div class="eyebrow">
        DEFENSIVE CYBERSECURITY PROJECT
    </div>

    <div class="about-title">
        Phishing Email Detection
        & Awareness Dashboard
    </div>

    <div class="about-description">
        PHISHGUARD is a student-built defensive cybersecurity
        platform that combines explainable rule-based analysis,
        static URL inspection, sender analysis, attachment
        metadata analysis and optional machine learning.
    </div>


    <div class="about-grid">

        <div class="about-card">

            <div class="about-card-title">
                Detection
            </div>

            <div class="about-card-text">
                Identifies multiple phishing-related signals
                instead of relying on a single keyword.
            </div>

        </div>


        <div class="about-card">

            <div class="about-card-title">
                Explainability
            </div>

            <div class="about-card-text">
                Every assessment provides understandable
                findings and recommended actions.
            </div>

        </div>


        <div class="about-card">

            <div class="about-card-title">
                Privacy
            </div>

            <div class="about-card-text">
                The project uses synthetic data and stores
                analysis metadata rather than full email bodies.
            </div>

        </div>


        <div class="about-card">

            <div class="about-card-title">
                Defensive Design
            </div>

            <div class="about-card-text">
                URLs are never opened and attachments are
                never executed.
            </div>

        </div>

    </div>


    <div class="technology-card">

        <div class="section-title">
            Technology Stack
        </div>

        <div class="tech-list">

            <span>Python</span>
            <span>Gradio</span>
            <span>SQLite</span>
            <span>Scikit-learn</span>
            <span>TF-IDF</span>
            <span>Plotly</span>
            <span>NLP</span>
            <span>Cybersecurity</span>

        </div>

    </div>

</div>

"""


# ============================================================
# 27. PROFESSIONAL CSS
# ============================================================
#
# IMPORTANT:
# This is the major visual correction.
#
# The palette is deliberately contrast-first.
# Every component has an explicit foreground/background pair.
# ============================================================

CUSTOM_CSS = r"""

/* =========================================================
   GLOBAL DESIGN SYSTEM
   ========================================================= */

:root {

    --bg-main: #080d18;
    --bg-sidebar: #0c1424;
    --bg-surface: #111b2d;
    --bg-surface-2: #162238;
    --bg-surface-3: #1b2941;

    --bg-input: #0b1424;
    --bg-input-hover: #101d31;

    --text-primary: #f8fafc;
    --text-secondary: #d7e0ec;
    --text-muted: #a7b5c7;

    --border: #2a3a55;
    --border-light: #344865;

    --blue: #38bdf8;
    --blue-dark: #0ea5e9;

    --green: #22c55e;
    --green-light: #86efac;

    --amber: #f59e0b;
    --amber-light: #fcd34d;

    --red: #ef4444;
    --red-light: #fca5a5;

    --purple: #a78bfa;
    --purple-light: #c4b5fd;

    --shadow:
        0 12px 35px rgba(0, 0, 0, 0.28);

    --radius: 14px;
}


/* =========================================================
   PAGE
   ========================================================= */

body {

    background:
        radial-gradient(
            circle at top right,
            rgba(56, 189, 248, 0.08),
            transparent 30%
        ),
        var(--bg-main) !important;

    color: var(--text-primary) !important;

    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif !important;
}


/* =========================================================
   GRADIO ROOT
   ========================================================= */

.gradio-container {

    max-width: 1500px !important;

    margin: auto !important;

    background:
        linear-gradient(
            180deg,
            #080d18 0%,
            #0b1220 100%
        ) !important;

    color: var(--text-primary) !important;

    padding: 24px !important;
}


/* =========================================================
   ALL TEXT
   ========================================================= */

.gradio-container,
.gradio-container p,
.gradio-container span,
.gradio-container div,
.gradio-container label {

    color: var(--text-primary);
}


.gradio-container .prose,
.gradio-container .prose p {

    color: var(--text-secondary) !important;
}


.gradio-container .prose strong {

    color: var(--text-primary) !important;
}


/* =========================================================
   HEADINGS
   ========================================================= */

h1,
h2,
h3,
h4,
h5,
h6 {

    color: var(--text-primary) !important;

    letter-spacing: -0.02em;
}


h1 {

    font-size: 2.2rem !important;
}


h2 {

    font-size: 1.55rem !important;
}


/* =========================================================
   MARKDOWN
   ========================================================= */

.markdown-body,
.prose {

    color: var(--text-secondary) !important;
}


.markdown-body a,
.prose a {

    color: var(--blue) !important;
}


.markdown-body code,
.prose code {

    background: #0b1424 !important;

    color: #bae6fd !important;

    border: 1px solid var(--border);

    padding: 2px 5px;

    border-radius: 5px;
}


/* =========================================================
   LABELS
   ========================================================= */

label {

    color: var(--text-secondary) !important;

    font-weight: 600 !important;

    letter-spacing: 0.01em;
}


/* =========================================================
   INPUTS
   ========================================================= */

input,
textarea,
select,
.gr-input,
input[type="text"],
input[type="email"],
input[type="number"] {

    background: var(--bg-input) !important;

    color: var(--text-primary) !important;

    border: 1px solid var(--border-light) !important;

    border-radius: 10px !important;

    box-shadow: none !important;

    caret-color: var(--blue) !important;
}


input:hover,
textarea:hover,
select:hover {

    border-color: #46617f !important;

    background: var(--bg-input-hover) !important;
}


input:focus,
textarea:focus,
select:focus {

    border-color: var(--blue) !important;

    outline: 3px solid rgba(56, 189, 248, 0.14) !important;

    box-shadow:
        0 0 0 1px var(--blue) !important;
}


textarea {

    line-height: 1.65 !important;
}


/* =========================================================
   PLACEHOLDERS
   ========================================================= */

input::placeholder,
textarea::placeholder {

    color: #91a4ba !important;

    opacity: 1 !important;
}


/* =========================================================
   DROPDOWNS
   ========================================================= */

select {

    color: var(--text-primary) !important;

    background-color: var(--bg-input) !important;
}


/* =========================================================
   BUTTONS
   ========================================================= */

button {

    color: #ffffff !important;

    font-weight: 700 !important;

    border-radius: 10px !important;

    border: 1px solid transparent !important;

    transition:
        transform 0.15s ease,
        box-shadow 0.15s ease,
        background 0.15s ease !important;
}


button:hover {

    transform: translateY(-1px);

    box-shadow:
        0 8px 20px rgba(0, 0, 0, 0.25) !important;
}


button:focus {

    outline: 3px solid rgba(56, 189, 248, 0.18) !important;
}


/* =========================================================
   PRIMARY BUTTON
   ========================================================= */

.primary {

    background:
        linear-gradient(
            135deg,
            #0284c7,
            #0ea5e9
        ) !important;

    color: #ffffff !important;

    border-color: #38bdf8 !important;
}


.primary:hover {

    background:
        linear-gradient(
            135deg,
            #0369a1,
            #0284c7
        ) !important;
}


/* =========================================================
   SECONDARY BUTTON
   ========================================================= */

.secondary {

    background: #1a2940 !important;

    color: #f8fafc !important;

    border: 1px solid #3a506f !important;
}


.secondary:hover {

    background: #223450 !important;
}


/* =========================================================
   TABS
   ========================================================= */

button[role="tab"] {

    background: transparent !important;

    color: #b8c6d8 !important;

    border: none !important;

    font-weight: 700 !important;
}


button[role="tab"]:hover {

    color: #ffffff !important;

    background: #17243a !important;
}


button[role="tab"][aria-selected="true"] {

    color: #ffffff !important;

    background: #12304b !important;

    border-bottom: 2px solid #38bdf8 !important;
}


/* =========================================================
   BLOCKS
   ========================================================= */

.block {

    background: var(--bg-surface) !important;

    border:
        1px solid var(--border) !important;

    border-radius: var(--radius) !important;

    box-shadow: var(--shadow) !important;
}


/* =========================================================
   DATAFRAME / TABLE
   ========================================================= */

table {

    background: var(--bg-surface) !important;

    color: var(--text-primary) !important;

    border-collapse: collapse !important;
}


thead,
th {

    background: #1b2a42 !important;

    color: #f8fafc !important;

    font-weight: 700 !important;

    border-color: var(--border-light) !important;
}


tbody,
tr,
td {

    background: var(--bg-surface) !important;

    color: #e5edf7 !important;

    border-color: var(--border) !important;
}


tbody tr:hover td {

    background: #182840 !important;

    color: #ffffff !important;
}


/* =========================================================
   FILE UPLOAD
   ========================================================= */

.file-preview,
.upload-container {

    background: var(--bg-input) !important;

    border: 1px dashed #46617f !important;

    color: var(--text-secondary) !important;
}


/* =========================================================
   HEADER
   ========================================================= */

.app-header {

    padding: 28px 30px;

    margin-bottom: 22px;

    border-radius: 18px;

    background:
        linear-gradient(
            135deg,
            #101d31 0%,
            #13243b 100%
        );

    border:
        1px solid #2e4565;

    box-shadow: var(--shadow);
}


.brand-row {

    display: flex;

    align-items: center;

    gap: 18px;
}


.brand-icon {

    width: 54px;

    height: 54px;

    display: flex;

    align-items: center;

    justify-content: center;

    border-radius: 14px;

    background:
        linear-gradient(
            135deg,
            #0284c7,
            #38bdf8
        );

    color: #ffffff;

    font-size: 25px;

    font-weight: 900;

    box-shadow:
        0 10px 25px rgba(
            14,
            165,
            233,
            0.22
        );
}


.brand-title {

    color: #ffffff;

    font-size: 29px;

    font-weight: 850;

    letter-spacing: -0.035em;
}


.brand-subtitle {

    color: #b8c8da;

    font-size: 14px;

    margin-top: 5px;
}


/* =========================================================
   EYEBROW
   ========================================================= */

.eyebrow {

    color: #67d5ff;

    font-size: 11px;

    font-weight: 800;

    letter-spacing: 0.16em;

    text-transform: uppercase;

    margin-bottom: 8px;
}


/* =========================================================
   KPI CARDS
   ========================================================= */

.overview-grid {

    display: grid;

    grid-template-columns:
        repeat(
            5,
            minmax(0, 1fr)
        );

    gap: 15px;

    margin: 18px 0 25px;
}


.kpi-card {

    position: relative;

    overflow: hidden;

    padding: 20px;

    min-height: 140px;

    border-radius: 14px;

    background: #121d30;

    border: 1px solid #2a3b57;

    box-shadow: var(--shadow);
}


.kpi-card::before {

    content: "";

    position: absolute;

    left: 0;

    top: 0;

    bottom: 0;

    width: 4px;
}


.kpi-card.blue::before {

    background: #38bdf8;
}


.kpi-card.red::before {

    background: #ef4444;
}


.kpi-card.amber::before {

    background: #f59e0b;
}


.kpi-card.green::before {

    background: #22c55e;
}


.kpi-card.purple::before {

    background: #a78bfa;
}


.kpi-label {

    color: #9fb0c4;

    font-size: 11px;

    font-weight: 800;

    letter-spacing: 0.11em;
}


.kpi-number {

    color: #ffffff;

    font-size: 34px;

    font-weight: 850;

    margin-top: 13px;
}


.kpi-description {

    color: #aebed0;

    font-size: 13px;

    margin-top: 3px;
}


/* =========================================================
   ANALYZER PANEL
   ========================================================= */

.analyzer-panel {

    padding: 22px;

    background: #111c2f;

    border:
        1px solid #2c3f5d;

    border-radius: 16px;

    box-shadow: var(--shadow);
}


.panel-title {

    color: #ffffff;

    font-size: 20px;

    font-weight: 800;

    margin-bottom: 5px;
}


.panel-description {

    color: #aebed0;

    font-size: 13px;

    line-height: 1.6;

    margin-bottom: 20px;
}


/* =========================================================
   RESULT AREA
   ========================================================= */

.result-container {

    display: flex;

    flex-direction: column;

    gap: 15px;

    margin-top: 15px;
}


.risk-hero {

    display: flex;

    align-items: center;

    justify-content: space-between;

    gap: 20px;

    padding: 25px;

    background:
        linear-gradient(
            135deg,
            #101c2f,
            #14243b
        );

    border:
        1px solid #314966;

    border-radius: 16px;
}


.risk-title {

    color: #ffffff;

    font-size: 24px;

    font-weight: 850;
}


.risk-subtitle {

    color: #aebfd2;

    max-width: 650px;

    margin-top: 6px;

    line-height: 1.55;
}


.risk-score-box {

    min-width: 155px;

    padding: 18px;

    text-align: center;

    border-radius: 14px;

    background: #0a1424;

    border: 1px solid #344a68;
}


.risk-score-label {

    color: #93a8be;

    font-size: 10px;

    font-weight: 800;

    letter-spacing: 0.15em;
}


.risk-score-number {

    color: #ffffff;

    font-size: 42px;

    font-weight: 900;

    line-height: 1.1;

    margin-top: 8px;
}


.risk-score-number span {

    color: #9fb0c4;

    font-size: 15px;

    font-weight: 600;
}


/* =========================================================
   CLASSIFICATION
   ========================================================= */

.classification-wrapper {

    width: 100%;
}


.classification {

    display: inline-flex;

    align-items: center;

    justify-content: center;

    padding: 10px 16px;

    border-radius: 999px;

    font-weight: 850;

    font-size: 13px;

    letter-spacing: 0.04em;

    border: 1px solid;
}


.classification.high {

    background: #39171b;

    color: #ffb4b4;

    border-color: #b93640;
}


.classification.suspicious {

    background: #3a2b0e;

    color: #fbd77c;

    border-color: #a8750a;
}


.classification.moderate {

    background: #172d3d;

    color: #8fdcff;

    border-color: #2778a0;
}


.classification.low {

    background: #103020;

    color: #91efad;

    border-color: #23864a;
}


/* =========================================================
   MINI SCORE GRID
   ========================================================= */

.mini-grid {

    display: grid;

    grid-template-columns:
        repeat(
            4,
            minmax(0, 1fr)
        );

    gap: 12px;
}


.mini-card {

    background: #121f33;

    border:
        1px solid #2b3e5b;

    border-radius: 12px;

    padding: 16px;
}


.mini-label {

    color: #9dafc3;

    font-size: 10px;

    font-weight: 800;

    letter-spacing: 0.12em;
}


.mini-value {

    color: #ffffff;

    font-size: 23px;

    font-weight: 800;

    margin-top: 7px;
}


/* =========================================================
   SECTION CARDS
   ========================================================= */

.section-card {

    padding: 21px;

    background: #111c2e;

    border:
        1px solid #2b3e5b;

    border-radius: 15px;

    box-shadow:
        0 8px 25px rgba(
            0,
            0,
            0,
            0.18
        );
}


.section-title {

    color: #ffffff;

    font-size: 17px;

    font-weight: 800;
}


.section-subtitle {

    color: #a8bacd;

    font-size: 13px;

    line-height: 1.6;

    margin-top: 5px;

    margin-bottom: 16px;
}


/* =========================================================
   FINDINGS
   ========================================================= */

.finding-row {

    display: flex;

    align-items: center;

    justify-content: space-between;

    gap: 15px;

    padding: 13px 0;

    border-bottom:
        1px solid #24354d;
}


.finding-row:last-child {

    border-bottom: none;
}


.finding-left {

    display: flex;

    align-items: center;

    gap: 12px;
}


.finding-icon {

    width: 30px;

    height: 30px;

    flex-shrink: 0;

    display: flex;

    align-items: center;

    justify-content: center;

    border-radius: 9px;

    background: #1c2c43;

    color: #67d5ff;

    font-weight: 900;

    border: 1px solid #35506e;
}


.finding-type {

    color: #ffffff;

    font-weight: 750;

    font-size: 14px;
}


.finding-description {

    color: #aebed0;

    font-size: 12px;

    margin-top: 3px;
}


/* =========================================================
   SEVERITY BADGES
   ========================================================= */

.severity-badge {

    display: inline-flex;

    align-items: center;

    justify-content: center;

    min-width: 70px;

    padding: 5px 9px;

    border-radius: 999px;

    font-size: 9px;

    font-weight: 850;

    letter-spacing: 0.08em;

    border: 1px solid;
}


.severity-badge.high {

    background: #39171b;

    color: #ffb4b4;

    border-color: #a9343e;
}


.severity-badge.medium {

    background: #3b2b0e;

    color: #fbd77c;

    border-color: #9b710d;
}


.severity-badge.low {

    background: #182b3c;

    color: #8fdcff;

    border-color: #2b6e92;
}


.severity-badge.info {

    background: #162b22;

    color: #8ce7aa;

    border-color: #2b784b;
}


/* =========================================================
   URL CARDS
   ========================================================= */

.url-card {

    background: #0b1525;

    border:
        1px solid #2a3e5b;

    border-radius: 11px;

    padding: 15px;

    margin-top: 10px;
}


.url-address {

    color: #8edfff;

    font-family:
        ui-monospace,
        SFMono-Regular,
        Consolas,
        monospace;

    font-size: 12px;

    overflow-wrap: anywhere;
}


.url-score {

    color: #b2c1d2;

    font-size: 12px;

    margin-top: 7px;
}


.url-score strong {

    color: #ffffff;
}


.url-card ul {

    color: #cbd7e5;

    line-height: 1.7;

    font-size: 12px;
}


/* =========================================================
   RECOMMENDATIONS
   ========================================================= */

.recommendation-card {

    background:
        linear-gradient(
            135deg,
            #101f31,
            #12263a
        );

    border-color: #2d536c;
}


.recommendation {

    display: flex;

    align-items: center;

    gap: 10px;

    color: #dbe8f5;

    padding: 9px 0;

    font-size: 13px;
}


.check {

    display: inline-flex;

    width: 22px;

    height: 22px;

    align-items: center;

    justify-content: center;

    flex-shrink: 0;

    border-radius: 50%;

    background: #123722;

    color: #7cf0a3;

    border: 1px solid #2d8650;

    font-weight: 900;
}


/* =========================================================
   DISCLAIMER
   ========================================================= */

.analysis-disclaimer {

    padding: 13px 15px;

    border-radius: 10px;

    background: #182334;

    color: #b5c4d5;

    border:
        1px solid #31445f;

    font-size: 11px;

    line-height: 1.55;
}


.analysis-disclaimer strong {

    color: #ffffff;
}


/* =========================================================
   EMPTY STATE
   ========================================================= */

.empty-result {

    padding: 50px 25px;

    text-align: center;

    background: #111c2e;

    border:
        1px solid #2b3e5b;

    border-radius: 15px;
}


.empty-icon {

    width: 48px;

    height: 48px;

    margin: auto;

    display: flex;

    align-items: center;

    justify-content: center;

    border-radius: 50%;

    background: #1a2c43;

    color: #67d5ff;

    border: 1px solid #365573;

    font-size: 20px;

    font-weight: 900;
}


.empty-title {

    color: #ffffff;

    font-size: 17px;

    font-weight: 800;

    margin-top: 15px;
}


.empty-text {

    color: #a9bbcf;

    font-size: 13px;

    margin-top: 7px;
}


/* =========================================================
   AWARENESS
   ========================================================= */

.awareness-header {

    padding: 24px;

    background:
        linear-gradient(
            135deg,
            #102238,
            #142b40
        );

    border:
        1px solid #2d5069;

    border-radius: 16px;

    margin-bottom: 16px;
}


.awareness-title {

    color: #ffffff;

    font-size: 27px;

    font-weight: 850;
}


.awareness-description {

    color: #b1c3d5;

    margin-top: 7px;
}


.lesson-grid {

    display: grid;

    grid-template-columns:
        repeat(
            3,
            minmax(0, 1fr)
        );

    gap: 14px;
}


.lesson-card {

    padding: 21px;

    background: #121e31;

    border:
        1px solid #2b3e5b;

    border-radius: 14px;

    min-height: 185px;
}


.lesson-number {

    color: #55cfff;

    font-size: 11px;

    font-weight: 900;

    letter-spacing: 0.1em;
}


.lesson-title {

    color: #ffffff;

    font-size: 17px;

    font-weight: 800;

    margin-top: 12px;
}


.lesson-description {

    color: #aebed0;

    font-size: 13px;

    line-height: 1.7;

    margin-top: 8px;
}


.checklist-card {

    margin-top: 16px;

    padding: 21px;

    background: #111c2e;

    border:
        1px solid #2b3e5b;

    border-radius: 14px;
}


.checklist-grid {

    display: grid;

    grid-template-columns:
        repeat(
            2,
            minmax(0, 1fr)
        );

    gap: 11px;

    color: #d6e2ef;

    font-size: 13px;

    margin-top: 15px;
}


/* =========================================================
   ABOUT
   ========================================================= */

.about-container {

    padding: 25px;
}


.about-title {

    color: #ffffff;

    font-size: 32px;

    font-weight: 900;

    max-width: 850px;

    letter-spacing: -0.035em;
}


.about-description {

    color: #b4c4d5;

    max-width: 850px;

    line-height: 1.75;

    margin-top: 12px;

    font-size: 14px;
}


.about-grid {

    display: grid;

    grid-template-columns:
        repeat(
            2,
            minmax(0, 1fr)
        );

    gap: 14px;

    margin-top: 22px;
}


.about-card {

    padding: 19px;

    background: #121e31;

    border:
        1px solid #2b3e5b;

    border-radius: 13px;
}


.about-card-title {

    color: #ffffff;

    font-size: 16px;

    font-weight: 800;
}


.about-card-text {

    color: #aebed0;

    font-size: 13px;

    line-height: 1.65;

    margin-top: 7px;
}


.technology-card {

    padding: 20px;

    margin-top: 15px;

    background: #101b2d;

    border:
        1px solid #2b3e5b;

    border-radius: 13px;
}


.tech-list {

    display: flex;

    flex-wrap: wrap;

    gap: 8px;

    margin-top: 14px;
}


.tech-list span {

    background: #1a2b42;

    color: #cfe0ef;

    border:
        1px solid #38516f;

    border-radius: 999px;

    padding: 7px 11px;

    font-size: 11px;

    font-weight: 700;
}


/* =========================================================
   RESPONSIVE
   ========================================================= */

@media (max-width: 1100px) {

    .overview-grid {

        grid-template-columns:
            repeat(
                3,
                minmax(0, 1fr)
            );
    }

    .lesson-grid {

        grid-template-columns:
            repeat(
                2,
                minmax(0, 1fr)
            );
    }

}


@media (max-width: 800px) {

    .gradio-container {

        padding: 12px !important;
    }

    .overview-grid {

        grid-template-columns:
            repeat(
                2,
                minmax(0, 1fr)
            );
    }

    .mini-grid {

        grid-template-columns:
            repeat(
                2,
                minmax(0, 1fr)
            );
    }

    .risk-hero {

        flex-direction: column;

        align-items: flex-start;
    }

    .risk-score-box {

        width: 100%;
    }

    .lesson-grid {

        grid-template-columns: 1fr;
    }

    .about-grid {

        grid-template-columns: 1fr;
    }

}


@media (max-width: 560px) {

    .overview-grid {

        grid-template-columns: 1fr;
    }

    .mini-grid {

        grid-template-columns: 1fr;
    }

    .checklist-grid {

        grid-template-columns: 1fr;
    }

    .brand-title {

        font-size: 22px;
    }

    .about-title {

        font-size: 25px;
    }

}


/* =========================================================
   SCROLLBARS
   ========================================================= */

* {

    scrollbar-width: thin;

    scrollbar-color:
        #3b516d
        #0b1424;
}


::-webkit-scrollbar {

    width: 9px;

    height: 9px;
}


::-webkit-scrollbar-track {

    background: #0b1424;
}


::-webkit-scrollbar-thumb {

    background: #3b516d;

    border-radius: 999px;
}


::-webkit-scrollbar-thumb:hover {

    background: #4c6685;
}

"""


# ============================================================
# 28. BUILD GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    css=CUSTOM_CSS,
    title="PHISHGUARD | Phishing Detection & Awareness"
) as demo:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.HTML(
        """
        <div class="app-header">

            <div class="brand-row">

                <div class="brand-icon">
                    P
                </div>

                <div>

                    <div class="brand-title">
                        PHISHGUARD
                    </div>

                    <div class="brand-subtitle">
                        Phishing Email Detection & Awareness Dashboard
                    </div>

                </div>

            </div>

        </div>
        """
    )


    # --------------------------------------------------------
    # OVERVIEW
    # --------------------------------------------------------

    with gr.Tab(
        "Dashboard"
    ):

        dashboard_html = gr.HTML(
            value=dashboard_overview()
        )

        gr.Markdown(
            """
            ## Security Operations Overview

            Analyze synthetic or authorized email samples using
            explainable sender, content, URL and attachment signals.
            """
        )

        with gr.Row():

            with gr.Column(
                scale=1
            ):

                classification_plot = gr.Plot(
                    value=make_classification_chart(),
                    label="Classification Distribution"
                )

            with gr.Column(
                scale=1
            ):

                risk_plot = gr.Plot(
                    value=make_risk_chart(),
                    label="Risk Distribution"
                )

        trend_plot = gr.Plot(
            value=make_trend_chart(),
            label="Detection Trend"
        )


        refresh_button = gr.Button(
            "↻ Refresh Dashboard",
            variant="secondary"
        )


    # --------------------------------------------------------
    # EMAIL ANALYZER
    # --------------------------------------------------------

    with gr.Tab(
        "Email Analyzer"
    ):

        gr.HTML(
            """
            <div class="analyzer-panel">

                <div class="panel-title">
                    Email Threat Analyzer
                </div>

                <div class="panel-description">
                    Enter a synthetic or authorized email sample.
                    The system performs static defensive analysis
                    without opening links or executing attachments.
                </div>

            </div>
            """
        )

        with gr.Row():

            with gr.Column():

                sender_input = gr.Textbox(
                    label="Sender Email",
                    placeholder="security@example.org",
                    lines=1
                )

                subject_input = gr.Textbox(
                    label="Email Subject",
                    placeholder="Enter the email subject",
                    lines=1
                )

                attachment_input = gr.Textbox(
                    label="Attachment Filename",
                    placeholder="Example: invoice.pdf",
                    lines=1
                )

            with gr.Column():

                body_input = gr.Textbox(
                    label="Email Body",
                    placeholder="Paste a safe synthetic email body here...",
                    lines=12
                )


        with gr.Row():

            analyze_button = gr.Button(
                "🔍 ANALYZE EMAIL",
                variant="primary",
                scale=2
            )

            phishing_demo_button = gr.Button(
                "Load Phishing Demo",
                variant="secondary"
            )

            legitimate_demo_button = gr.Button(
                "Load Legitimate Demo",
                variant="secondary"
            )


        analysis_output = gr.HTML(
            value="""
            <div class="empty-result">

                <div class="empty-icon">
                    P
                </div>

                <div class="empty-title">
                    Ready for Analysis
                </div>

                <div class="empty-text">
                    Submit an email sample to generate an
                    explainable phishing risk assessment.
                </div>

            </div>
            """
        )


        analyze_button.click(
            fn=analyze_for_dashboard,
            inputs=[
                sender_input,
                subject_input,
                body_input,
                attachment_input
            ],
            outputs=analysis_output
        )


        phishing_demo_button.click(
            fn=load_phishing_demo,
            outputs=[
                sender_input,
                subject_input,
                body_input,
                attachment_input
            ]
        )


        legitimate_demo_button.click(
            fn=load_legitimate_demo,
            outputs=[
                sender_input,
                subject_input,
                body_input,
                attachment_input
            ]
        )


    # --------------------------------------------------------
    # ANALYSIS HISTORY
    # --------------------------------------------------------

    with gr.Tab(
        "Analysis History"
    ):

        gr.Markdown(
            """
            ## Analysis History

            The dashboard stores analysis metadata and findings
            for demonstration and security analytics.
            Full email bodies are not stored by default.
            """
        )

        history_output = gr.Dataframe(
            value=history_table(),
            interactive=False,
            wrap=True
        )

        refresh_history = gr.Button(
            "↻ Refresh History",
            variant="secondary"
        )

        refresh_history.click(
            fn=history_table,
            outputs=history_output
        )


    # --------------------------------------------------------
    # AWARENESS
    # --------------------------------------------------------

    with gr.Tab(
        "Security Awareness"
    ):

        gr.HTML(
            value=awareness_html()
        )


    # --------------------------------------------------------
    # PROJECT
    # --------------------------------------------------------

    with gr.Tab(
        "Project"
    ):

        gr.HTML(
            value=ABOUT_HTML
        )


    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.HTML(
        """
        <div style="
            margin-top:28px;
            padding:18px;
            border-top:1px solid #263750;
            text-align:center;
            color:#91a4ba;
            font-size:11px;
            line-height:1.6;
        ">

            <strong style="color:#dce8f5;">
                PHISHGUARD
            </strong>

            &nbsp;•&nbsp;

            Defensive Cybersecurity Education Project

            <br>

            Synthetic / authorized data only.
            Suspicious URLs are analyzed statically and are
            never opened by this application.

        </div>
        """
    )


    # --------------------------------------------------------
    # REFRESH DASHBOARD
    # --------------------------------------------------------

    def refresh_dashboard():

        return (
            dashboard_overview(),
            make_classification_chart(),
            make_risk_chart(),
            make_trend_chart()
        )


    refresh_button.click(
        fn=refresh_dashboard,
        outputs=[
            dashboard_html,
            classification_plot,
            risk_plot,
            trend_plot
        ]
    )


# ============================================================
# 29. AUTOMATED TESTS
# ============================================================

def run_security_tests():

    results = []

    def test(
        test_id,
        scenario,
        condition
    ):

        results.append({
            "Test ID": test_id,
            "Scenario": scenario,
            "Result": "PASS" if condition else "FAIL"
        })


    # 1
    result = analyze_email(
        "training@example.org",
        "Workshop Reminder",
        "This is a normal workshop reminder.",
        ""
    )

    test(
        "T01",
        "Legitimate email",
        result["risk_result"]["score"] <= 40
    )


    # 2
    result = analyze_email(
        "security-alert@example.net",
        "URGENT Verify",
        "Act immediately. Verify your password.",
        ""
    )

    test(
        "T02",
        "Urgent phishing-style email",
        result["risk_result"]["score"] > 20
    )


    # 3
    result = analyze_email(
        "support@example.net",
        "Account",
        "Please verify your password immediately.",
        ""
    )

    test(
        "T03",
        "Credential request",
        result["content_result"]["score"] > 0
    )


    # 4
    result = analyze_email(
        "billing@example.net",
        "Payment",
        "Your invoice payment is due.",
        ""
    )

    test(
        "T04",
        "Financial request",
        result["content_result"]["score"] > 0
    )


    # 5
    result = analyze_email(
        "user@example.org",
        "Hello",
        "Dear customer, welcome.",
        ""
    )

    test(
        "T05",
        "Generic greeting",
        any(
            f["type"] == "Generic Greeting"
            for f in result["findings"]
        )
    )


    # 6
    url_result = analyze_url(
        "https://example.org/news"
    )

    test(
        "T06",
        "Safe example URL",
        url_result["score"] < 20
    )


    # 7
    url_result = analyze_url(
        "http://198.51.100.10/verify"
    )

    test(
        "T07",
        "Raw IP URL",
        url_result["score"] >= 20
    )


    # 8
    url_result = analyze_url(
        "http://example.org/login"
    )

    test(
        "T08",
        "Non-HTTPS URL",
        url_result["score"] >= 15
    )


    # 9
    url_result = analyze_url(
        "https://a.b.c.example.org/login"
    )

    test(
        "T09",
        "Excessive subdomains",
        url_result["score"] > 0
    )


    # 10
    url_result = analyze_url(
        "https://example.org/verify-password"
    )

    test(
        "T10",
        "Suspicious URL keyword",
        url_result["score"] > 0
    )


    # 11
    result = analyze_email(
        "user@example.org",
        "Hello",
        "Normal message.",
        ""
    )

    test(
        "T11",
        "No URL",
        result["url_result"]["score"] == 0
    )


    # 12
    result = analyze_email(
        "user@example.org",
        "Links",
        "Visit https://example.org and https://example.net.",
        ""
    )

    test(
        "T12",
        "Multiple URLs",
        len(result["url_result"]["results"]) == 2
    )


    # 13
    attachment = analyze_attachment(
        "meeting.pdf"
    )

    test(
        "T13",
        "Normal attachment",
        attachment["score"] == 0
    )


    # 14
    attachment = analyze_attachment(
        "update.exe"
    )

    test(
        "T14",
        "Executable attachment",
        attachment["score"] >= 25
    )


    # 15
    attachment = analyze_attachment(
        "invoice.pdf.exe"
    )

    test(
        "T15",
        "Double extension",
        attachment["score"] >= 25
    )


    # 16
    result = analyze_email(
        "user@example.org",
        "",
        "Normal body.",
        ""
    )

    test(
        "T16",
        "Empty subject",
        any(
            f["type"] == "Subject"
            for f in result["findings"]
        )
    )


    # 17
    result = analyze_email(
        "user@example.org",
        "Test",
        "",
        ""
    )

    test(
        "T17",
        "Empty body",
        any(
            f["type"] == "Body"
            for f in result["findings"]
        )
    )


    # 18
    sender = analyze_sender(
        "invalid-address"
    )

    test(
        "T18",
        "Invalid sender",
        sender["score"] > 0
    )


    # 19
    result = analyze_email(
        "user@example.org",
        "URGENT!!!",
        "THIS IS A VERY URGENT MESSAGE!!! ACT NOW!!!",
        ""
    )

    test(
        "T19",
        "High uppercase ratio",
        result["content_result"]["score"] > 0
    )


    # 20
    result = analyze_email(
        "user@example.org",
        "Hello!!!",
        "Important!!! Please read!!!",
        ""
    )

    test(
        "T20",
        "Multiple exclamation marks",
        any(
            f["type"] == "Formatting"
            for f in result["findings"]
        )
    )


    # 21
    result = calculate_phishing_score(
        {
            "score": 100,
            "findings": []
        },
        {
            "score": 100,
            "findings": []
        },
        {
            "score": 100,
            "results": []
        },
        {
            "score": 100,
            "findings": []
        }
    )

    test(
        "T21",
        "Risk score capped at 100",
        result["score"] <= 100
    )


    # 22
    result = analyze_email(
        "database-test@example.org",
        "Database Test",
        "Testing database persistence.",
        ""
    )

    conn = sqlite3.connect(
        DB_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        "SELECT COUNT(*) FROM analyses"
    )

    count = cursor.fetchone()[0]

    conn.close()

    test(
        "T22",
        "Database save",
        count > 0
    )


    # 23
    test(
        "T23",
        "API-style input validation",
        isinstance(
            analyze_email(
                "",
                "",
                "",
                ""
            ),
            dict
        )
    )


    # 24
    test(
        "T24",
        "ML model availability",
        ML_AVAILABLE
    )


    # 25
    test(
        "T25",
        "Analysis history retrieval",
        isinstance(
            history_table(),
            pd.DataFrame
        )
    )


    return pd.DataFrame(
        results
    )


test_results = run_security_tests()

test_results.to_csv(
    TEST_DIR / "automated_test_results.csv",
    index=False
)


# ============================================================
# 30. CREATE PROJECT README
# ============================================================

README = """
# PHISHGUARD
## Phishing Email Detection & Awareness Dashboard

PHISHGUARD is a defensive cybersecurity education project that
analyzes synthetic or authorized email samples and generates
explainable phishing risk assessments.

## Features

- Sender analysis
- Social-engineering language detection
- URL static analysis
- Attachment filename analysis
- Explainable risk scoring
- Synthetic phishing dataset
- TF-IDF machine learning model
- SQLite analysis history
- Security analytics
- Awareness training
- Automated testing
- Professional SOC-style dashboard

## Safety

This project:

- Does not send phishing emails
- Does not collect credentials
- Does not create credential-harvesting pages
- Does not execute attachments
- Does not automatically visit suspicious URLs
- Uses synthetic or authorized data

## Technology

Python
Gradio
SQLite
Scikit-learn
TF-IDF
Plotly
Pandas
NumPy

## Disclaimer

This project is designed for cybersecurity education and
defensive analysis using synthetic or authorized data.
"""

(BASE_DIR / "README.md").write_text(
    README,
    encoding="utf-8"
)


# ============================================================
# 31. FINAL STARTUP INFORMATION
# ============================================================

print("=" * 70)
print("PHISHGUARD - Phishing Email Detection & Awareness Dashboard")
print("=" * 70)

print()
print("Project directory:")
print(BASE_DIR)

print()
print("Synthetic dataset:")
print(DATASET_PATH)

print()
print("Dataset records:")
print(len(dataset))

print()
print("ML metrics:")
print(ML_METRICS)

print()
print("Automated tests:")
print(
    f"{(test_results['Result'] == 'PASS').sum()}/"
    f"{len(test_results)} passed"
)

print()
print("Starting dashboard...")
print("=" * 70)


# ============================================================
# 32. LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=False,
    show_error=True
)

PHISHGUARD - Phishing Email Detection & Awareness Dashboard

Project directory:
/content/phishguard

Synthetic dataset:
/content/phishguard/data/phishing_email_dataset.csv

Dataset records:
600

ML metrics:
{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

Automated tests:
25/25 passed

Starting dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a0c5c56cbcdfe8df6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
